# Multi-Model YOLO Validation and Reporting

Run the `run_yolo_validation_report.py` script on multiple YOLO models, collect metrics, and compare results.


In [ ]:

# 1. Set Up Environment and Install Dependencies
import os
import sys
from pathlib import Path

import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image as IPImage

# Configure matplotlib for notebook
%matplotlib inline
matplotlib.rcParams['figure.max_open_warning'] = 50

print(f"Python: {sys.version}")
print(f"Torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
else:
    # Running locally
    BASE_DIR = Path.cwd().parent


PROJECT_ROOT = BASE_DIR
SCRIPT_PATH = PROJECT_ROOT / "yolo_test" / "run_yolo_validation_report.py"

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "yolo_test") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "yolo_test"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Script path: {SCRIPT_PATH}")


# Dataset Selection
# Option 1: Full dataset (~100k images) - for final optimization: "bdd100k_yolo"
# Option 2: Limited dataset (representative samples) - for quick tuning: "bdd100k_yolo_limited"
DATASET_NAME = 'bdd100k_yolo_limited'
DATASET_SPLT = 'test'  # 'train', 'val', or 'test'

BATCH_SIZE = 64

# 2. Import validation functions from script
from run_yolo_validation_report import run_validation_pipeline, visualize_predictions
print("✓ Successfully imported validation functions")

# 3. Method Loop over models, run validation, and collect metrics
results_summary = []
validation_results = {}

def test_model(models_configs):
    
    for cfg in models_configs:
        print("=" * 80)
        print(f"Running model: {cfg['name']} | dataset={cfg['dataset']} | split={cfg['split']} | IoU={cfg['iou']}")
        print("=" * 80)
        
        try:
            result = run_validation_pipeline(
                model_name=cfg["name"],
                dataset_name=DATASET_NAME,
                split=DATASET_SPLT,
                iou_threshold=cfg["iou"],
                base_dir=PROJECT_ROOT,
                use_wandb=True,
                save_reports=True,
                batch_size=BATCH_SIZE,
            )
            
            validation_results[cfg["name"]] = result
            
            overall = result["metrics"]["overall"]
            yolo_overall = result["metrics"]["yolo_metrics"]
            
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "precision_confusion": overall["precision"],
                "recall_confusion": overall["recall"],
                "f1_confusion": overall["f1"],
                "precision_yolo": yolo_overall["precision"],
                "recall_yolo": yolo_overall["recall"],
                "map50": yolo_overall["map50"],
                "map50_95": yolo_overall["map50_95"],
                "params_m": result["model_info"]["params"] / 1e6,
                "size_mb": result["model_info"]["size(MB)"],
                "fps": result["metrics"]["fps"],
                "status": "ok",
                "run_dir": str(result["run_dir"]),
            })
            
        except Exception as e:
            print(f"⚠️ Model {cfg['name']} failed: {e}")
            results_summary.append({
                "model_name": cfg["name"],
                "dataset": cfg["dataset"],
                "split": cfg["split"],
                "iou": cfg["iou"],
                "status": "error",
            })






In [ ]:
# 4. Select model configurations to test

MODEL_CONFIGS = [
    {"name": "yolov8n",  "iou": 0.5}
]


test_model(MODEL_CONFIGS)


In [ ]:
MODEL_CONFIGS = [
{"name": "yolov9s", "iou": 0.5},
]

test_model(MODEL_CONFIGS)


In [ ]:
results_df = pd.DataFrame(results_summary)
results_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

success_df = results_df[results_df["status"] == "ok"].copy()

# ---------------------------
# Assign stable colors
# ---------------------------
palette = sns.color_palette("tab10", len(success_df))
model_colors = {
    m: palette[i]
    for i, m in enumerate(success_df["model_name"])
}

models = success_df["model_name"].tolist()
colors = [model_colors[m] for m in models]

sns.set_style("whitegrid")
plt.close('all')

fig, axes = plt.subplots(2, 2, figsize=(18, 12), dpi=120)

# -----------------------------------
# 1) mAP@0.5
# -----------------------------------
axes[0,0].bar(models, success_df["map50"], color=colors)
axes[0,0].set_title("mAP@0.5", fontweight="bold")
axes[0,0].set_ylabel("mAP50")
axes[0,0].tick_params(axis='x', rotation=45)
for i, v in enumerate(success_df["map50"]):
    axes[0,0].text(i, v * 1.02, f"{v:.3f}", ha="center")

# -----------------------------------
# 2) F1 Score
# -----------------------------------
axes[0,1].bar(models, success_df["f1_confusion"], color=colors)
axes[0,1].set_title("F1 Score", fontweight="bold")
axes[0,1].set_ylabel("F1")
axes[0,1].tick_params(axis='x', rotation=45)
for i, v in enumerate(success_df["f1_confusion"]):
    axes[0,1].text(i, v * 1.02, f"{v:.3f}", ha="center")

# -----------------------------------
# 3) Size vs Performance (scatter)
# -----------------------------------
for i, row in success_df.iterrows():
    axes[1,0].scatter(row["size_mb"], row["map50"], 
                      s=200, color=model_colors[row["model_name"]])
    axes[1,0].annotate(row["model_name"],
                       (row["size_mb"], row["map50"]),
                       xytext=(5, 5), textcoords="offset points")

axes[1,0].set_title("Model Size vs mAP50", fontweight="bold")
axes[1,0].set_xlabel("Size (MB)")
axes[1,0].set_ylabel("mAP50")
axes[1,0].grid(True, alpha=0.3)

# -----------------------------------
# 4) FPS (tiny values → use relative label placement)
# -----------------------------------
axes[1,1].bar(models, success_df["fps"], color=colors)
axes[1,1].set_title("FPS (Inference Speed)", fontweight="bold")
axes[1,1].set_ylabel("FPS")
axes[1,1].tick_params(axis='x', rotation=45)

for i, v in enumerate(success_df["fps"]):
    axes[1,1].text(i, v * 1.10, f"{v:.4f}", ha="center")

plt.tight_layout()
plt.show()


In [ ]:

# Create a consistent color for each model
unique_models = success_df["model_name"].unique()
colors = plt.cm.tab10(range(len(unique_models)))   # Choose a colormap
color_map = dict(zip(unique_models, colors))       # model → color

metrics = {
    "mAP@0.5": success_df["map50"],
    "F1 Score": success_df["f1_confusion"],
    "Model Size (MB)": success_df["size_mb"],
    "FPS": success_df["fps"]
}

for title, values in metrics.items():
    plt.figure(figsize=(10, 5))

    # Apply consistent colors
    bar_colors = [color_map[m] for m in success_df["model_name"]]

    plt.bar(success_df["model_name"], values, color=bar_colors)

    plt.title(title, fontweight="bold", fontsize=18)
    plt.xlabel("Model Name", fontweight="normal", fontsize=14)
    plt.ylabel(title, fontweight="normal", fontsize=14)
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
# 8. Per-Class Performance Comparison
if validation_results:
    print("=" * 80)
    print("Per-Class Performance Comparison Across Models")
    print("=" * 80)
    
    # Collect per-class data
    per_class_comparison = []
    for model_name, result in validation_results.items():
        df_metrics = result["df_metrics"]
        for _, row in df_metrics.iterrows():
            per_class_comparison.append({
                "model": model_name,
                "class": row["Class"],
                "precision": row["Precision"],
                "recall": row["Recall"],
                "f1": row["F1-Score"],
                "map50": row["mAP@0.5"],
            })
    
    per_class_df = pd.DataFrame(per_class_comparison)
    
    # Get unique classes
    classes = per_class_df["class"].unique()
    
    # Plot comparison for each metric
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    
    for ax, metric in zip(axes.flatten(), ["precision", "recall", "f1", "map50"]):
        pivot_data = per_class_df.pivot(index="class", columns="model", values=metric)
        pivot_data.plot(kind="bar", ax=ax, width=0.8)
        ax.set_title(f"{metric.upper()} by Class", fontweight='bold', fontsize=14)
        ax.set_ylabel(metric.capitalize())
        ax.set_xlabel("Class")
        ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display detailed table
    print("\n📊 Detailed Per-Class Metrics:")
    display(per_class_df.pivot_table(
        index="class",
        columns="model",
        values=["precision", "recall", "f1", "map50"],
        aggfunc="first"
    ).round(4))

In [ ]:
# 9. Summary Report
print("=" * 80)
print("SUMMARY REPORT")
print("=" * 80)

if not results_df.empty:
    success_df = results_df[results_df["status"] == "ok"]
    
    if not success_df.empty:
        # Best model by different criteria
        best_map50 = success_df.loc[success_df["map50"].idxmax()]
        best_f1 = success_df.loc[success_df["f1_confusion"].idxmax()]
        best_fps = success_df.loc[success_df["fps"].idxmax()]
        smallest = success_df.loc[success_df["size_mb"].idxmin()]
        
        print("\n🏆 Best Model by mAP@0.5:")
        print(f"   {best_map50['model_name']} - mAP@0.5: {best_map50['map50']:.4f}")
        
        print("\n🏆 Best Model by F1 Score:")
        print(f"   {best_f1['model_name']} - F1: {best_f1['f1_confusion']:.4f}")
        
        print("\n⚡ Fastest Model:")
        print(f"   {best_fps['model_name']} - FPS: {best_fps['fps']:.2f}")
        
        print("\n📦 Smallest Model:")
        print(f"   {smallest['model_name']} - Size: {smallest['size_mb']:.1f} MB")
        
        print("\n" + "=" * 80)
        print("Detailed Comparison Table:")
        print("=" * 80)
        display(success_df[[
            "model_name", "map50", "map50_95", "f1_confusion",
            "precision_yolo", "recall_yolo", "params_m", "size_mb", "fps"
        ]].round(4))
    else:
        print("No successful runs to summarize.")
else:
    print("No results to display.")